# Compute global temperature average for observational datasets

This notebook collects observational timeseries of Global Mean Surface Temperature, to be used in determining obserbed Global Warming Levels

The first version of this dataset applied a method used by the WMO, but this version does not follow this method exactly.

Most of those datasets are given as anomalies relative to some period. We transform all of them to anomalies relative to 1850-1900, and then add an estimate of the pre-industrial mean to get "absolute" timeseries. This dataset should only be used to compute anomalies, but giving absolute values gives values in the same range as for the climate models.

The value of 13.88°C used for the pre-industrial mean temperature comes from Berkeley Earth data.

## Preparation

In [ ]:
import re
import zipfile
from pathlib import Path

import dask
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import xarray as xr

import xscen as xs


dask.config.set(num_workers=12)

In [ ]:
# This was computed from Berkeley Earth and is an estimate of the 1850-1900 mean
# it is used to get series that looks like real temperature.
preind_abs = 13.88


def clean(tas, fill=False, ref=None, make_abs=False):
    """Transform anomalies relative to some period to anomalies relative to 1850-1900 and then adds an estimate for that period."""
    # Transform to anomalies in reference to 1850-1900
    tas = tas - tas.sel(year=slice(1850, 1900)).mean()

    # Fill 1850-1900 with 0s
    first_year = tas.year.values[0]
    last_0_year = min(1900, first_year)
    zeros = xr.DataArray(
        [0] * (last_0_year - 1850),
        dims=("year",),
        coords={"year": np.arange(1850, last_0_year)},
    )
    # Concat over years and add the estimate to get absolute values
    tas = xr.concat([zeros, tas], "year")

    # Add a pre-industrial value
    tas = tas + preind_abs
    return tas


# The output, a list of DataArrays with a year dim and a "source" singleton dim
temps = {}

### Berkeley Earth
Rohde, R. A.; Hausfather, Z. The Berkeley Earth Land/Ocean Temperature Record. Earth System Science Data 2020, 12, 3469–3479. https://doi.org/10.5194/essd-12-3469-2020.

The annual summary of the Global Monthly Averages from 1850 to preset is available online as text file. It comes as an anomaly relative to the 1951-1980 average, which is given directly in the header of the text file. We have the choice of using temperature above or below sea ice for the ocean component. I didn't see anything in the WMO recommendation about this, so we will use the temperature above.

For this dataset, we will make two versions. A "LowRes" one with the global annual summaries from a 1°x1° product that was updated until Q2 2025. The newer version comes from a higher-resolution product, at 0.25°x0.25° but is still listed as "experimental" on their website.

The main difference between the two is on the pre-industrial period, which yields significantly different anomalies.

In [ ]:
# Get data
file = Path("Berkeley_data_old.txt")

with file.open("wb") as f:
    res = requests.get(
        "https://berkeley-earth-temperature.s3.us-west-1.amazonaws.com/Global/Land_and_Ocean_summary.txt",
        timeout=15,
    )
    f.write(res.content)

df = pd.read_table(
    file,
    skiprows=58,
    usecols=[0, 1],
    names=["year", "temp"],
    sep=r"\s+",
    index_col="year",
)
da = df.temp.to_xarray().assign_attrs(units="°C")

# Get global average for the reference period of the data
with file.open("r") as f:
    for line in f:
        if "% Estimated " in line:
            data = re.search(r"(\d{2}.\d{3})", next(f))
            break
ref_avg = float(data.groups()[0])

da_berk = da + ref_avg

temps["Berkeley-LowRes"] = da_berk.expand_dims(source=["Berkeley-LowRes"])

In [ ]:
# Get data
file = Path("Berkeley_data.txt")

with file.open("wb") as f:
    res = requests.get(
        "https://storage.googleapis.com/berkeley-earth-temperature-hr/global/Global_TAVG_annual.txt",
        timeout=15,
    )
    f.write(res.content)

df = pd.read_table(
    file,
    skiprows=82,
    usecols=[0, 1],
    names=["year", "temp"],
    sep=r"\s+",
    index_col="year",
)
da = df.temp.to_xarray().assign_attrs(units="°C")

# Get global average for the reference period of the data
with file.open("r") as f:
    for line in f:
        if "% Estimated " in line:
            data = re.search(r"(\d{2}.\d{3})", next(f))
            break
ref_avg = float(data.groups()[0])

da_berk = da + ref_avg

temps["Berkeley-HighRes"] = da_berk.expand_dims(source=["Berkeley-HighRes"])

In [ ]:
temps["Berkeley-HighRes"].sel(year=slice(1850, 1900)).mean("year")

### GISTEMP
GISTEMP Team, 2023: GISS Surface Temperature Analysis (GISTEMP), version 4. NASA Goddard Institute for Space Studies. Dataset accessed 2023-12-06 at https://data.giss.nasa.gov/gistemp/.

This one was not used by the IPCC AR6 WG1 Chap 1. This dataset comes in a CSV of anomalies relative to 1951-1980.

In [ ]:
df = pd.read_csv(
    "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv",
    usecols=["Year", "J-D"],
    skiprows=1,
    index_col="Year",
    na_values="***",
)
da = df["J-D"].to_xarray().rename(Year="year").rename("temp").assign_attrs(units="°C")

# Need to be "cleaned" as it doesn't go to 1850, it's anomalies with another period and we don't have a reference value
da_giss = clean(da)

temps["GISTEMP"] = da_giss.expand_dims(source=["GISTEMPv4"])

### HadCRUT5
Morice, C. P., Kennedy, J. J., Rayner, N. A., Winn, J. P., Hogan, E., Killick, R. E., et al. (2021). An updated assessment of near-surface temperature change from 1850: the HadCRUT5 data set. Journal of Geophysical Research: Atmospheres, 126, e2019JD032361. https://doi.org/10.1029/2019JD032361

The CSV is an anomaly relative to 1961-1990.

In [ ]:
df = pd.read_csv(
    "https://www.metoffice.gov.uk/hadobs/hadcrut5/data/HadCRUT.5.1.0.0/analysis/diagnostics/HadCRUT.5.1.0.0.analysis.summary_series.global.annual.csv",
    usecols=[0, 1],
    index_col=0,
)
da = df["Anomaly (deg C)"].to_xarray().rename(Time="year").rename("temp").assign_attrs(units="°C")

# Needed as data are anomalies without reference and to another period
da_crut = clean(da)

temps["HadCRUT5"] = da_crut.expand_dims(source=["HadCRUT5"])

## Kadow et al.

Kadow, C., Hall, D.M. & Ulbrich, U. Artificial intelligence reconstructs missing climate information. Nat. Geosci. 13, 408–413 (2020). https://doi.org/10.1038/s41561-020-0582-5

The original netCDF used in the IPCC AR6 WG1 Report is downloaded from the publisher. It was based on HadCRUT4. The updated versions are provided here :https://zenodo.org/records/19630991.

In [ ]:
file = Path("Kadow2020.zip")

with file.open("wb") as f:
    res = requests.get(
        "https://static-content.springer.com/esm/art%3A10.1038%2Fs41561-020-0582-5/MediaObjects/41561_2020_582_MOESM5_ESM.zip",
        timeout=15,
    )
    f.write(res.content)

zipfile.ZipFile("Kadow2020.zip").extract("Fig4/20crAI-HAD-GM.nc")

In [ ]:
ds = xr.open_dataset("Fig4/20crAI-HAD-GM.nc")
da = ds.tas.assign_attrs(units="°C")
da = da.resample(time="YS").mean()
da = da.assign_coords(time=da.time.dt.year).rename(time="year")
da = xs.spatial_mean(da.to_dataset(), method="cos-lat").tas
da = clean(da)

In [ ]:
temps["Kadow2020"] = da.expand_dims(source=["Kadow2020"])

In [ ]:
file = Path("Kadow2026.nc")

with file.open("wb") as f:
    res = requests.get(
        "https://zenodo.org/records/19630991/files/Kadow_et_al_2026_HadCRUT.5.1.0.0.AIinfilled.anomalies.ensemble_global_annual_mean_185001-202512.nc?download=1",
        timeout=15,
    )
    f.write(res.content)

ds = xr.open_dataset("Kadow2026.nc")
da = ds.tas.assign_attrs(units="°C")
da = da.resample(time="YS").mean()
da = da.assign_coords(time=da.time.dt.year).rename(time="year")
da = xs.spatial_mean(da.to_dataset(), method="cos-lat").tas
da = clean(da)

In [ ]:
temps["Kadow2026"] = da.expand_dims(source=["Kadow2026"])

### NOAAGlobalTemp v6.1
Huang, Boyin; Yin, Xungang; Menne, Matthew J.; and Vose, Russell S. 2026. NOAA Global Surface Temperature Dataset (NOAAGlobalTemp), Version 6.1. Annual average Land-Ocean. NOAA National Centers for Environmental Information. https://doi.org/10.25921/vvaa-wq11. Accessed May 12th 2026.


Available as a text file as an anomaly relative to 1971-2000. The files are updated each month with only the last month kept available.

In [ ]:
df = pd.read_table(
    "https://www.ncei.noaa.gov/data/noaa-global-surface-temperature/v5.1/access/timeseries/aravg.ann.land_ocean.90S.90N.v5.1.0.202312.asc",
    sep=r"\s+",
    usecols=[0, 1],
    index_col=0,
    names=["year", "temp"],
)

da = df.temp.to_xarray().assign_attrs(units="°C")

da_noaa = clean(da)

temps["NOAAv5"] = da_noaa.expand_dims(source=["NOAAGlobalTempv5"])

In [ ]:
df = pd.read_table(
    "https://www.ncei.noaa.gov/data/noaa-global-surface-temperature/v6.1/access/timeseries/aravg.ann.land_ocean.90S.90N.v6.1.0.202604.asc",
    sep=r"\s+",
    usecols=[0, 1],
    index_col=0,
    names=["year", "temp"],
)

da = df.temp.to_xarray().assign_attrs(units="°C")

da_noaa = clean(da)

temps["NOAA"] = da_noaa.expand_dims(source=["NOAAGlobalTempv6"])

## Combine all

In [ ]:
ds = xr.concat(temps.values(), "source", join="outer")
ds

**Don't forget most timeseries were aligned so that they are 13.88C on 1850-1900**! (except the Berkeley ones)

In [ ]:
# A figure to look at it
fig, ax = plt.subplots(figsize=(10, 3))  # noqa
ds.plot(ax=ax, hue="source")
ax.set_title("Global Average Temperature - Ob")
ax.set_xlabel("years")
ax.set_ylabel("[°C]")

In [ ]:
# A figure to look at it
fig, ax = plt.subplots(figsize=(10, 3))  # noqa
ds.rolling(year=30, center=True, min_periods=20).mean().plot(ax=ax, hue="source")
ax.set_title("Global Average Temperature - Ob")
ax.set_xlabel("years")
ax.set_ylabel("[°C]")

In [ ]:
# A figure to look at it, but now aligned on another reference period, and smoothed

fig, ax = plt.subplots(figsize=(10, 3))  # noqa
(ds - ds.sel(year=slice(1850, 1900)).mean("year")).rolling(year=30, center=True, min_periods=20).mean().plot(ax=ax, hue="source")
ax.set_title("Global Average Temperature Anomalies vs 1850-1900 - Obs")
ax.set_xlabel("years")
ax.set_ylabel("[°C]")

**Same as IPCC AR6 WG1 Report, Chap. 1, Fig 1.12.** :

In [ ]:
(ds.sel(year=slice(1986, 2005)).mean("year") - ds.sel(year=slice(1850, 1900)).mean("year")).sel(
    source=["Berkeley-LowRes", "HadCRUT5", "Kadow2020", "NOAAGlobalTempv5"]
).mean()

**Same datasets as above, but updated in May 2026** :

In [ ]:
(ds.sel(year=slice(1986, 2005)).mean("year") - ds.sel(year=slice(1850, 1900)).mean("year")).sel(
    source=["Berkeley-HighRes", "HadCRUT5", "Kadow2026", "NOAAGlobalTempv6"]
).mean()

In [ ]:
db = xr.open_dataset("../src/xscen/data/IPCC_annual_global_tas.nc", engine="h5netcdf")
db

In [ ]:
ds2 = (
    ds.assign_coords(year=pd.to_datetime(ds.year, format="%Y"))
    .rename(year="time")
    .drop_vars("source")
    .rename(source="simulation")
    .assign_coords(
        source=(("simulation",), ds.source.values),
        data_source=(
            ("simulation",),
            ["Converted to anomalies to PI and added 13.88°C as PI mean."] * ds.source.size,
        ),
        mip_era=(("simulation",), ["obs"] * ds.source.size),
        experiment=(("simulation",), ["obs"] * ds.source.size),
        member=(("simulation",), [""] * ds.source.size),
    )
    .rename("tas")
    .to_dataset()
)
ds2

In [ ]:
# Remove previous version of Obs.
dbsim = db.where(db.mip_era != "obs", drop=True)
db2 = xr.concat([dbsim, ds2], "simulation", join="outer")
db2.attrs["description"] = (
    "Data comes from different sources as indicated by the 'data_source' coordinate. "
    "IPCC Atlas data comes from the *_tas_landsea files of "
    "https://github.com/IPCC-WG1/Atlas/tree/main/datasets-aggregated-regionally/data, column 'world'. "
    "Extra simulations where added by computing the cos(lat)-weighted global average "
    "of the annually-averaged monthly tas data available on the ESGF nodes. "
    "Observational datasets were also added, along with older versions to get the same ensemble as used by the IPCC AR6 WG1 report"
)

In [ ]:
db2.to_netcdf("../src/xscen/data/IPCC_annual_global_tas_withObs.nc")

In [ ]:
for exp in pd.unique(db2.experiment.values):
    fig, ax = plt.subplots()
    db2.where(db2.experiment == exp, drop=True).tas.plot(hue="simulation", add_legend=False)
    ax.set_title(exp)